In [0]:
spark.sql("USE CATALOG relp")
spark.sql("USE SCHEMA bronze")
spark.sql("USE SCHEMA silver")

In [0]:
import importlib
import pipeline.silver.transformation as transformation

# Reload updated code
importlib.reload(transformation)
from pipeline.silver.transformation import clean_transactions

In [0]:
print("=" * 60)
print("🚀 SILVER LAYER STARTED")
print("=" * 60)

# 1️⃣ Read Bronze
df_bronze = spark.table("bronze.bronze_transactions")

bronze_count = df_bronze.count()
print(f"\n📊 Bronze Count: {bronze_count}")

# 2️⃣ Apply transformation
df_silver = clean_transactions(df_bronze)

# 3️⃣ Preview (clean)
print("\n📊 Silver Preview:")
display(
    df_silver.select(
        "transaction_id",
        "city",
        "deal_price",
        "commission_amount",
        "payment_mode",
        "deal_status",
        "year_month"
    ).limit(10)
)

# 4️⃣ Write Silver table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.silver_transactions")

# 5️⃣ Validation (single count only)
silver_count = df_silver.count()

print("\n📊 Validation:")
print(f"Bronze Count: {bronze_count}")
print(f"Silver Count: {silver_count}")
print(f"Rows Dropped: {bronze_count - silver_count}")

# 6️⃣ Table check
print("\n📂 Silver Tables:")
display(spark.sql("SHOW TABLES IN silver"))